# WSP on Colab

Select a hosted GPU runtime. The download cell prepares the public Sketchy v1 dataset in the current VM.

In [ ]:
from pathlib import Path
import subprocess

CONTENT = Path("/content")
if not CONTENT.is_dir():
    raise RuntimeError("Select a hosted Colab runtime.")
REPO = CONTENT / "WSP"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/HieuNM1804/WSP.git", str(REPO)], check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)

In [ ]:
import sys

SKETCHY = CONTENT / "Sketchy"
subprocess.run([sys.executable, str(REPO / "wsp_sbir/download_sketchy.py"), "--destination", str(SKETCHY)], check=True)

## Prepare WSP

Setup checks the sandbox, reuses Colab’s GPU PyTorch, and installs a CPU environment for sandboxed Python.

In [ ]:
import os

subprocess.run(["bash", str(REPO / "wsp_sbir/setup_colab.sh"), str(SKETCHY)], check=True)
CODEX = REPO / ".colab-tools/node_modules/.bin/codex"

## Run WSP

Sign in to Codex on this runtime before starting a run. The optional login cell requests a device code. Set the forward-pass budget and leave `resume_name` empty for a new run. Set `publish_to_github = True` before launching if Git already has write access to the destination repository.

In [ ]:
login = False
if login:
    subprocess.run([str(CODEX), "login", "--device-auth"], check=True)
else:
    subprocess.run([str(CODEX), "login", "status"], check=False)

In [ ]:
from datetime import datetime, timezone

dataset = "sketchy_2"
budget = 2
resume_name = None
publish_to_github = False
dataset_slug = dataset.replace("_", "")
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
name = resume_name or f"astra-max-{dataset_slug}-{run_id}"
run = REPO / "wsp_sbir/runs" / name
bundle = run / "result.zip"
launcher_env = dict(os.environ, WSP_GPU_PYTHON=sys.executable, WSP_CODEX_BIN=str(CODEX),
                    WSP_SANDBOX_VENV=str(REPO / "wsp_sbir/.venv_sandbox"), WSP_SANDBOX_PROC="0")
subprocess.run([str(CODEX), "login", "status"], check=True)
if resume_name:
    if not (run / "state.json").is_file():
        raise FileNotFoundError(run / "state.json")
    args = ["--run", str(run), "--resume"]
else:
    args = ["--task", f"tasks/{dataset}", "--run", str(run), "--budget", str(budget)]
print(f"Run: {name}", flush=True)
subprocess.run(["bash", str(REPO / "wsp_sbir/run_local.sh"), *args], check=True, env=launcher_env)
if publish_to_github:
    subprocess.run([sys.executable, str(REPO / "wsp_sbir/publish_results.py"),
                    "--from-bundle", str(bundle), "--push"], cwd=REPO, check=True)

## Download the result bundle

The run cell creates `result.zip` as soon as the run ends. Download it before closing the runtime if publishing from your local Git checkout. In VS Code, use the Colab file explorer to download the path printed by the run.

In [ ]:
from google.colab import files

files.download(str(bundle))